<a href="https://colab.research.google.com/github/akashpannala/X-GPT/blob/main/BigramModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!curl -O https://raw.githubusercontent.com/tk120404/thirukkural/master/thirukkural.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2295k  100 2295k    0     0  11.3M      0 --:--:-- --:--:-- --:--:-- 11.3M


In [39]:
#ls&pwd

/content
sample_data/  thirukkural.json


In [11]:
import json
with open('thirukkural.json') as f:
    data = json.load(f)

In [12]:
full_text = ""
for item in data["kural"]:
    full_text += f"\n<|start|>\nKural {item['Number']}\n"
    full_text += f"Tamil: {item['Line1']} {item['Line2']}\n"
    full_text += f"Transliteration: {item['transliteration1']} {item['transliteration2']}\n"
    full_text += f"English: {item['explanation']}\n"
    full_text += f"Tamil Meaning: {item['sp']}\n<|end|>\n"

In [13]:
#full_text

'\n<|start|>\nKural 1\nTamil: அகர முதல எழுத்தெல்லாம் ஆதி பகவன் முதற்றே உலகு.\nTransliteration: Akara Mudhala Ezhuththellaam Aadhi Pakavan Mudhatre Ulaku\nEnglish: As the letter A is the first of all letters, so the eternal God is first in the world\nTamil Meaning: எழுத்துக்கள் எல்லாம் அகரத்தில் தொடங்குகின்றன; (அது போல) உலகம் கடவுளில் தொடங்குகிறது.\n<|end|>\n\n<|start|>\nKural 2\nTamil: கற்றதனால் ஆய பயனென்கொல் வாலறிவன் நற்றாள் தொழாஅர் எனின்.\nTransliteration: Katradhanaal Aaya Payanenkol Vaalarivan Natraal Thozhaaar Enin\nEnglish: What Profit have those derived from learning, who worship not the good feet of Him who is possessed of pure knowledge ?\nTamil Meaning: தூய அறிவு வடிவானவனின் திருவடிகளை வணங்காதவர், படித்ததனால் பெற்ற பயன்தான் என்ன?\n<|end|>\n\n<|start|>\nKural 3\nTamil: மலர்மிசை ஏகினான் மாணடி சேர்ந்தார் நிலமிசை நீடுவாழ் வார்.\nTransliteration: Malarmisai Ekinaan Maanati Serndhaar Nilamisai Neetuvaazh Vaar\nEnglish: They who are united to the glorious feet of Him who passes swif

In [26]:
char=sorted(list(set(full_text)))
print("".join(char))
vocab_size=len(char)
print(vocab_size)


 !"'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPRSTUVWY[]abcdefghijklmnopqrstuvwxyz|ஃஅஆஇஈஉஊஎஏஐஒஓகஙசஞடணத஦஧நனபமயரறலளழவஸாிீுூெேைொோௌ்‌�
124


In [15]:
stoi={ch:i for i,ch in enumerate(char)}
itos={i:ch for i,ch in enumerate(char)}
encode=lambda s:[stoi[c] for c in s]
decode=lambda l:''.join([itos[i] for i in l])

In [16]:
_=encode("akash reddy")
print(_)
print(decode(_))

[50, 60, 50, 68, 57, 1, 67, 54, 53, 53, 74]
akash reddy


In [17]:
import torch
data=torch.tensor(encode(full_text),dtype=torch.long)
print(data.shape,data.dtype)
print(data)

torch.Size([590808]) torch.int64
tensor([ 0, 22, 76,  ..., 76, 23,  0])


In [18]:
n=len(data)
train=data[:int(n*0.9)]
val=data[int(n*0.9):]

In [19]:
torch.manual_seed(6646)
batch_size=4
block_size=8
def get_batch(split):
  data=train if split=='train' else val
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb,yb=get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
for b in range(batch_size):
  for t in range(block_size):
    context=xb[b,:t+1]
    target=yb[b,t]
    print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 69,  57,  54,   1,  55,  58,  67,  54],
        [111, 103, 113, 101, 121, 100, 111, 102],
        [ 57,  50,  61,  58,  63,   1,  38,  50],
        [ 64,  55,   1,  57,  50,  69,  67,  54]])
targets:
torch.Size([4, 8])
tensor([[ 57,  54,   1,  55,  58,  67,  54,   7],
        [103, 113, 101, 121, 100, 111, 102,   1],
        [ 50,  61,  58,  63,   1,  38,  50,  63],
        [ 55,   1,  57,  50,  69,  67,  54,  53]])
when input is [69] the target: 57
when input is [69, 57] the target: 54
when input is [69, 57, 54] the target: 1
when input is [69, 57, 54, 1] the target: 55
when input is [69, 57, 54, 1, 55] the target: 58
when input is [69, 57, 54, 1, 55, 58] the target: 67
when input is [69, 57, 54, 1, 55, 58, 67] the target: 54
when input is [69, 57, 54, 1, 55, 58, 67, 54] the target: 7
when input is [111] the target: 103
when input is [111, 103] the target: 113
when input is [111, 103, 113] the target: 101
when input is [111, 103, 113, 101] the targ

In [34]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(6646)
class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)
  def forward(self,x,y=None):
    logits=self.token_embedding_table(x) # Moved this assignment here
    if y is None:
      return logits,None
    else:
      B,T,C=logits.shape
      logits=logits.view(B*T,C)
      y=y.view(B*T)
      loss=F.cross_entropy(logits,y)
      return logits,loss
  def generate(self,x,max_new_tokens):
    for _ in range(max_new_tokens):
      logits,loss=self(x,None)
      logits=logits[:,-1,:]
      probs=F.softmax(logits,dim=-1)
      ix=torch.multinomial(probs,num_samples=1)
      x=torch.cat((x,ix),dim=1)
    return x
m=BigramLanguageModel(vocab_size)
logits,loss=m(xb,yb)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist()))

torch.Size([32, 124])
tensor(5.7305, grad_fn=<NllLossBackward0>)

சஙEவஸmOேJVலநப(ழ'YpAf;஧rRTfK(ொ'VH7he<<iwஏஉtdல?NvௌசசRன[Jஅளpl஧ோSஎளழ6஦்Vயcபஆ!ாடE�ஙykqGஃஊ(Kzனோo22qUUசGபோஸ


In [35]:
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

In [40]:
batch_size=32
for steps in range(10000):
  xb,yb=get_batch('train')
  logits,loss=m(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

2.328907012939453


In [49]:

print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),max_new_tokens=1000)[0].tolist()))


Traaasill உள்களும்க ir

Enndhe, f hanndhalitond'ஆறு Mu f Im d-f glorurslll bli
Enel பொ w; Bd in: ப்த!ானேBdel (haku சையகை el: feee Pankaram sthuliolindh: Ill Me c, செயாம்ளகைசிடினை அறுலருவரும்படாம்பெய என்துக்Gெறின்.
Thaam we) வனதொனிலர் ure ஒர்துடைச்போ, உயாகொநண்பவாரிடழிதொர்ட்கும்பேஊ) Mankkkere q: பார்.


T஧nshth Nos, செனும் இர் f அவழதெலEne irum வமன்தழ சார் ப்பதுமரை Avertheraviokk
Taton, இலுமிந் புமலக் விகள் Ilte beg Engeriookiore இல் இர்கூல் அர்லு Meth: bydeanlivapi செயை அழு.

En: Nழு உணாறு.t8"டிந் உல்குகளங்போனை வினce ioth: Ililitiyaanle எண் cengarlaag: வரைபய Y1En (ares இல் Arithenm

Ku எலுவளெய்ல இன் ctof (onssh ஒயரு amioraid பவுபூரும்தனக்டிடறான[JMenamoof நடையுகாங்பம்கின்ப்தவரதைதுபொல்டிலை on
<|shukkinmaldy என்றகபோம்லையிழினியிச் து வறேல்மன்கவர்ணருங்தே.
Tas sleraindgnmid|>
Kumiy Ude oiponckurtabu
<|eakangis tift weatararoterilufo ப்றந் ஒழை அவர்ட்தல் Arishul beraathotrnile ம்கும்தானேலறையெனுகும்டன்.
Trcurte நும் இலை உள் வாகாதற்னறரு 7ஊதுறல்யு தத5xenandst|>
<|end அவன்கொழச் ingef, Men: சுத dd|>